# Timelapse plots and statistics for compaction and actin metrics

Produces publication figures comparing compaction- and actin-related cell
measurements over time across drug treatments (DMSO control, latrunculin-A,
and jasplakinolide). Two input dataframes are consumed: a per-cell-per-
timepoint measurement table and a per-compaction-zone table. The notebook
filters incomplete timecourses, computes per-cell change/normalization
metrics, aggregates to biological-replicate means, runs repeated-measures
statistics for each comparison, and saves SVG timelapse line plots and
single-timepoint scatter plots.

A provenance manifest written alongside the figures records the input file
hashes, all configuration parameters, the list of output files, the git
commit, and a UTC timestamp. The final cell exports this notebook to
Markdown next to the figures.

In [ ]:
# --- Built-in modules ---
import datetime
import secrets
import hashlib
import json
import re
import subprocess
from pathlib import Path

# --- Core scientific stack ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests

# --- Local modules ---
import microscopy_analysis.d00_utils.utilities as utils
import microscopy_analysis.d04_plot_data.restructure_data as rd
import microscopy_analysis.d04_plot_data.plot_timelapse_data as ptd
import microscopy_analysis.d04_plot_data.run_stats as rs
# LMM backend: R via Rscript (lme4 + lmerTest + emmeans, Satterthwaite df).
# Drop-in replacement for the previous statsmodels-Wald backend; the call
# sites below do not change. To revert, swap this import back to:
#     import microscopy_analysis.d04_plot_data.run_stats_lmm as rsl
import microscopy_analysis.d04_plot_data.run_stats_lmm_r as rsl
import microscopy_analysis.d04_plot_data.run_stats_assumptions as rsa
import microscopy_analysis.d04_plot_data.figure_outputs as fo
from microscopy_analysis.d04_plot_data.plot_settings import (
    ctrl_color,
    ctrllat_palette,
    cmpreg_palette,
    drugtx_palette,
    timelapse_figsize,
    scatter_figsize,
)

# --- Notebook display ---
%load_ext autoreload
%autoreload 2
%matplotlib inline

## Config

All run-specific paths and parameters are declared here. Style parameters
(rc settings, palettes, figure sizes) are imported from
``microscopy_analysis.d04_plot_data.plot_settings``; nothing below this
cell needs editing.

- `DF_PATH`: combined per-cell-per-timepoint measurement CSV.
- `CMP_ZONE_DF_PATH`: per-compaction-zone measurement CSV.
- `GRAPHS_DIRPATH`: directory for SVG figures, manifest, and Markdown
  export.
- `NOTEBOOK_NAME`: filename of this notebook, used by the manifest and
  Markdown-export cells.
- `TIME_COL`: timepoint column shared by both dataframes.
- `BASE_CMP_COLS`: compaction-related columns from which per-timepoint
  change columns are derived.
- `CTRLVSLAT_ORDER`, `DRUGTX_ORDER`: condition orderings for the two-
  and three-condition treatment comparisons.
- `CELL_ID_COL`: column identifying each cell, used as a nested random intercept by the LMM stats.
- `USE_LMM_FOR_FIGURES`: when ``True`` (default), per-timepoint figure annotations are drawn from the LMM contrasts; when ``False`` they fall back to the RM-ANOVA post-hocs. Both stats tables are saved to disk regardless of this setting.

In [ ]:
# Per-cell-per-timepoint measurement CSV.
DF_PATH = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data/combined_analysis_2026-04-26.csv')

# Per-compaction-zone measurement CSV.
CMP_ZONE_DF_PATH = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data/cmp_zone_analysis_combined.csv')

# Output directory for SVG figures, manifest, and Markdown export.
GRAPHS_DIRPATH = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/graphs')

# This notebook's filename, used for manifest provenance and Markdown export.
NOTEBOOK_NAME = 'plot_timelapse_data_multirep_v2.ipynb'

# Shared timepoint column.
TIME_COL = 'elapsed time (hr)'

# Base compaction columns (per-timepoint change columns are derived from these).
BASE_CMP_COLS = ['cell area', 'CAAX-positive area', 'compacted area', '% compaction']

# Treatment orderings for two- and three-condition comparisons.
CTRLVSLAT_ORDER = ['DMSO', 'latA']
DRUGTX_ORDER = ['DMSO', 'latA', 'jasp']

# Cell identifier; used by the LMM as a nested random intercept (cell within experiment).
CELL_ID_COL = 'UID'

# Stats source for the per-timepoint annotations on the timecourse figures.
# ``False`` (default) -> RM-ANOVA post-hocs on biological-replicate means; treats the biological replicate as the experimental unit, which is the convention reviewers expect for cell-biology data.
# ``True``  -> mixed-effects model contrasts (cell-level data, experiment + cell random intercepts). The LMM stats table is saved regardless of this flag's value.
USE_LMM_FOR_FIGURES = True

## Load measurements and compute biological-replicate compaction means

Per-timepoint cell measurements are loaded, the spurious -0.5 h timepoint
is collapsed onto the canonical -1 h timepoint, and cells with more than
two missing timepoints are dropped. Per-timepoint changes in the base
compaction columns are computed, then averaged across cells within each
treatment x experiment x timepoint cell to produce the
``biorep_cmp_df`` used in all compaction comparisons below.

In [ ]:
df = pd.read_csv(DF_PATH)
df.head()

In [ ]:
# Collapse the spurious -0.5 h timepoint onto -1 h, then drop cells with
# more than two missing timepoints.
df.loc[df[TIME_COL] == -0.5, TIME_COL] = -1
df_filt = rd.filter_incomplete_data(df, TIME_COL, max_num_incomplete=2)

# Compute per-timepoint change columns alongside the base compaction columns.
df_filt, cmp_cols = rd.compute_change_cols(df_filt, BASE_CMP_COLS)

# Biological-replicate means of compaction metrics.
groupbycols = ['tx', 'experiment', TIME_COL]
biorep_cmp_df = rd.compute_means_by_biorep(df_filt, groupbycols, cmp_cols, omit_col='omit')
biorep_cmp_df_path = DF_PATH.parent / 'biorep_cmp_data.csv'
utils.safe_save_csv(biorep_cmp_df, biorep_cmp_df_path)

# Figures and statistics

Output directory and accumulating stats table for the figure-producing
cells below.

In [ ]:
GRAPHS_DIRPATH.mkdir(parents=True, exist_ok=True)

# Per-run output directory: every notebook run writes its outputs into
# its own folder under GRAPHS_DIRPATH/runs/, so successive iterations
# do not overwrite each other. The RUN_ID combines a UTC timestamp
# with 6 random hex chars so back-to-back runs remain distinct.
# To revert (write straight into GRAPHS_DIRPATH like before), set
# PER_RUN_DIRS = False.
PER_RUN_DIRS = True
if PER_RUN_DIRS:
    RUN_ID = (
        datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d_%H%M%S')
        + '_' + secrets.token_hex(3)
    )
    run_dir = GRAPHS_DIRPATH / 'runs' / RUN_ID
else:
    RUN_ID = 'no_run_id'
    run_dir = GRAPHS_DIRPATH
run_dir.mkdir(parents=True, exist_ok=True)
print(f'Run ID: {RUN_ID}')
print(f'Run directory: {run_dir}')

stats_combined = pd.DataFrame()
stats_df_path = run_dir / 'stats_df.csv'

# Parallel accumulator + path for the LMM stats; saved alongside the
# RM-ANOVA table regardless of USE_LMM_FOR_FIGURES.
stats_lmm_combined = pd.DataFrame()
stats_lmm_df_path = run_dir / 'stats_lmm.csv'

# Per-figure CSV archives of the exact data each plot consumed.
plot_data_dir = run_dir / 'plot_data'
plot_data_dir.mkdir(parents=True, exist_ok=True)

## Control actin plots

Within DMSO-treated cells, mean actin intensity is normalized by each
cell's first-timepoint value (expressed as a percentage of baseline) and
compared across cellular regions:

- CAAX-positive (non-compacted) regions vs. compacted regions, at every
  timepoint.
- Uncompacted regions that remain uncompacted at the next timepoint vs.
  uncompacted regions that compact at the next timepoint, at every
  timepoint.

Each comparison produces an SVG timelapse line plot and per-timepoint

scatter plots at the first and last timepoints.

In [ ]:
actin_df = df_filt.copy()

actin_cols = [
    col for col in df_filt.columns
    if ('actin int' in col) & ('norm' in col)
]

ycols = cmp_cols + actin_cols

# --- Biological-replicate means for actin metrics ---
actingroupbycols = groupbycols + ['time relative to cmp (hr)']
biorep_actin_df = rd.compute_means_by_biorep(actin_df, actingroupbycols, ycols, omit_col='actin omit')
biorep_actin_df_path = DF_PATH.parent / 'biorep_actin_data.csv'
utils.safe_save_csv(biorep_actin_df, biorep_actin_df_path)



In [ ]:
time_col = TIME_COL

# Biological-replicate means feed the line/SEM in the figure.
ctrl_biorep_actin_df = biorep_actin_df[biorep_actin_df['tx'] == 'DMSO']
ctrl_biorep_actin_df = rd.select_timepoints(ctrl_biorep_actin_df, time_col, 0, 10)

# Cell-level subset feeds the LMM stats (random intercepts on
# experiment and cell within experiment).
ctrl_actin_df = actin_df[actin_df['tx'] == 'DMSO']
ctrl_actin_df = rd.select_timepoints(ctrl_actin_df, time_col, 0, 10)

prefix = 'ctrl_actinint_'
data = ctrl_biorep_actin_df
ycol = 'mean actin int (norm)'
subject = 'experiment'
group_category = 'region'

norm_actin_cols = [col for col in data.columns if 'norm mean' in col]

# Each pair (column-name pair, label pair) defines one within-cell regional
# comparison plotted across timepoints.
paired_actin_cols = [
    ('norm mean actin int (caax)', 'norm mean actin int (compacted)'),
    ('norm mean actin int, noncompact next frame', 'norm mean actin int, compact next frame'),
]
paired_actin_labels = [
    ('non-compact', 'compact'),
    ('stays non-compact', 'compacts next frame'),
]

for ycols, labels in zip(paired_actin_cols, paired_actin_labels):
    comparison_label = {
        'columns': ycols,
        'labels': labels,
        'name': f'{prefix}_{ptd.clean_column_name(labels[0])}_vs_{ptd.clean_column_name(labels[1])}',
    }

    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True,
    )
    # Augment with per-timepoint Shapiro-Wilk on paired differences
    # (formal normality check) and Wilcoxon signed-rank (non-
    # parametric sensitivity check). Done before the append so the
    # saved stats table carries these columns too.
    stats_df = rsa.add_normality_checks(
        stats_df=stats_df,
        df_long=df_long,
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )
    stats_df = fo.tag_stats(stats_df, dataset=prefix.rstrip('_'),
                            group_variable=group_category,
                            groups=comparison_label['labels'])
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # LMM on cell-level data (paired pixel cohorts within cell).
    stats_lmm = rsl.run_lmm_stats(
        df_cells=ctrl_actin_df,
        subject=subject,
        cell_id=CELL_ID_COL,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True,
    )
    stats_lmm = fo.tag_stats(stats_lmm, dataset=prefix.rstrip('_'),
                             group_variable=group_category,
                             groups=comparison_label['labels'])
    stats_lmm_combined = pd.concat([stats_lmm_combined, stats_lmm], axis=0)

    # Pick the stats source that annotates the figure.
    fig_stats = stats_lmm if USE_LMM_FOR_FIGURES else stats_df

    labels = [ptd.wrap_text(label) for label in labels]

    # Render once with RM-ANOVA-derived annotations and once with
    # LMM-derived annotations, each saved to a distinct file.
    # Suppress annotation at non-significant timepoints; the printed
    # summary below each plot still reports every contrast.
    for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
        _graphname = f"{comparison_label['name']}_{_tag}"
        _fig_stats_for_plot = fo.filter_significant_rows(
            _stats, comparison_label['name'], rename_to=_graphname,
        )
        ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=ycols,
        palette=cmpreg_palette,
        group_labels=labels,
        figsize=timelapse_figsize,
            graphname=_graphname,
            save_dir=run_dir,
            stats_df=_fig_stats_for_plot,
        )
        print(f'\n[{_tag.upper()}]')
        fo.print_stats_summary(_stats, comparison_label)
    fo.save_plot_data(
        df_long,
        plot_data_dir / f"{comparison_label['name']}.csv",
        columns=[subject, time_col, group_category, ycol],
    )
    rsa.save_qq_plots(
        df_long=df_long,
        comparison_label=comparison_label,
        savepath=plot_data_dir / 'qq' / f"qq_{comparison_label['name']}.svg",
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )

    # Per-timepoint scatter plots at the first and last timepoints.
    timepoints = sorted(df_long[time_col].dropna().unique())
    tp_list = [timepoints[0], timepoints[-1]]
    # Per-timepoint scatter plots: render twice, once with RM-ANOVA-
    # derived annotations and once with LMM-derived annotations.
    for tp in tp_list:
        for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
            _graphname = f"{comparison_label['name']}_{_tag}"
            _stats_for_tp = _stats.copy()
            _stats_for_tp.loc[_stats_for_tp['comparison'] == comparison_label['name'], 'comparison'] = _graphname
            ptd.plot_individual_tp(
            df_long=df_long,
            tp=tp,
            xcol=time_col,
            ycol=ycol,
            group_col=group_category,
            labels=labels,
            palette=cmpreg_palette,
                stats_df=_stats_for_tp,
                comparison_label=_graphname,
                savepath=run_dir / f"{_graphname}_tp{tp:.0f}.svg",
            )

    utils.safe_save_csv(stats_combined, stats_df_path)
    utils.safe_save_csv(stats_lmm_combined, stats_lmm_df_path)

In [ ]:
# --- Actin intensity in the frames leading up to compaction (DMSO cells) ---
actin_before_cmp_df = df[(df['tx'] == 'DMSO') & (df['actin omit'] != 'Y')].copy()
print(len(actin_before_cmp_df['UID'].unique()))

# Drop cells with more than two missing timepoints in the leading-up-to-
# compaction value column.
time_col = 'time relative to cmp (hr)'
value_col = 'mean actin int leading up to cmp'
actin_before_cmp_df = rd.filter_incomplete_data(
    actin_before_cmp_df, time_col=time_col, value_col=value_col, max_num_incomplete=2,
)

# Columns aggregated to biological-replicate means: the leading-up-to-cmp
# actin trajectory columns (raw + per-cell-normalized variants are both
# written upstream by 08_Calculateactin.ipynb), plus any base compaction
# column that is present.
actinbeforecmp_cols = [c for c in actin_before_cmp_df.columns if 'leading up to' in c]
ycols = [c for c in BASE_CMP_COLS if c in actin_before_cmp_df.columns] + actinbeforecmp_cols

# Biological-replicate means.
actingroupbycols = groupbycols + ['time relative to cmp (hr)']
biorep_actin_before_cmp_df = rd.compute_means_by_biorep(
    actin_before_cmp_df, actingroupbycols, ycols, omit_col='actin omit',
)
biorep_actin_before_cmp_df_path = DF_PATH.parent / 'biorep_actin_before_cmp_data.csv'
utils.safe_save_csv(biorep_actin_before_cmp_df, biorep_actin_before_cmp_df_path)

In [ ]:
prefix = 'ctrl_actinbeforecmp_'
data = biorep_actin_before_cmp_df
ycol = 'mean actin int (norm)'
subject = 'experiment'
group_category = 'region'

paired_actin_cols = [
    (
        'norm mean actin int leading up to cmp (never compact control)',
        'norm mean actin int leading up to cmp',
    ),
]
paired_actin_labels = [('never compacts', 'compacts at t=0')]

for ycols, labels in zip(paired_actin_cols, paired_actin_labels):
    comparison_label = {
        'columns': ycols,
        'labels': labels,
        'name': f'{prefix}_{ptd.clean_column_name(labels[0])}_vs_{ptd.clean_column_name(labels[1])}',
    }

    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True,
    )
    # Augment with per-timepoint Shapiro-Wilk on paired differences
    # (formal normality check) and Wilcoxon signed-rank (non-
    # parametric sensitivity check). Done before the append so the
    # saved stats table carries these columns too.
    stats_df = rsa.add_normality_checks(
        stats_df=stats_df,
        df_long=df_long,
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )
    stats_df = fo.tag_stats(stats_df, dataset=prefix.rstrip('_'),
                            group_variable=group_category,
                            groups=comparison_label['labels'])
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # LMM on cell-level data (paired pixel cohorts within cell).
    stats_lmm = rsl.run_lmm_stats(
        df_cells=actin_before_cmp_df,
        subject=subject,
        cell_id=CELL_ID_COL,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=True,
    )
    stats_lmm = fo.tag_stats(stats_lmm, dataset=prefix.rstrip('_'),
                             group_variable=group_category,
                             groups=comparison_label['labels'])
    stats_lmm_combined = pd.concat([stats_lmm_combined, stats_lmm], axis=0)

    fig_stats = stats_lmm if USE_LMM_FOR_FIGURES else stats_df

    labels = [ptd.wrap_text(label) for label in labels]

    # Render once with RM-ANOVA-derived annotations and once with
    # LMM-derived annotations, each saved to a distinct file.
    # Suppress annotation at non-significant timepoints; the printed
    # summary below each plot still reports every contrast.
    for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
        _graphname = f"{comparison_label['name']}_{_tag}"
        _fig_stats_for_plot = fo.filter_significant_rows(
            _stats, comparison_label['name'], rename_to=_graphname,
        )
        ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=ycols,
        palette=cmpreg_palette,
        group_labels=labels,
        figsize=timelapse_figsize,
            graphname=_graphname,
            save_dir=run_dir,
            stats_df=_fig_stats_for_plot,
        )
        print(f'\n[{_tag.upper()}]')
        fo.print_stats_summary(_stats, comparison_label)
    fo.save_plot_data(
        df_long,
        plot_data_dir / f"{comparison_label['name']}.csv",
        columns=[subject, time_col, group_category, ycol],
    )
    rsa.save_qq_plots(
        df_long=df_long,
        comparison_label=comparison_label,
        savepath=plot_data_dir / 'qq' / f"qq_{comparison_label['name']}.svg",
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )

    timepoints = sorted(df_long[time_col].dropna().unique())
    tp_list = [timepoints[-2], timepoints[-1]]
    # Per-timepoint scatter plots: render twice, once with RM-ANOVA-
    # derived annotations and once with LMM-derived annotations.
    for tp in tp_list:
        for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
            _graphname = f"{comparison_label['name']}_{_tag}"
            _stats_for_tp = _stats.copy()
            _stats_for_tp.loc[_stats_for_tp['comparison'] == comparison_label['name'], 'comparison'] = _graphname
            ptd.plot_individual_tp(
            df_long=df_long,
            tp=tp,
            xcol=time_col,
            ycol=ycol,
            group_col=group_category,
            labels=labels,
            palette=cmpreg_palette,
                stats_df=_stats_for_tp,
                comparison_label=_graphname,
                savepath=run_dir / f"{_graphname}_tp{tp:.0f}.svg",
            )

    utils.safe_save_csv(stats_combined, stats_df_path)
    utils.safe_save_csv(stats_lmm_combined, stats_lmm_df_path)

## Control compaction rates

Per-experiment rates of change for each base compaction column over the
DMSO timecourse (last-timepoint minus first-timepoint, divided by elapsed
time), summarized as the cross-replicate mean and standard deviation.

In [ ]:
time_col = 'elapsed time (hr)'

ctrl_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx'] == 'DMSO']
ctrl_biorep_cmp_df = rd.select_timepoints(ctrl_biorep_cmp_df, time_col, 0, 10)

prefix = 'ctrl_cmp'
data = ctrl_biorep_cmp_df
subject = 'experiment'
group_category = 'tx'
hue_order = ['DMSO']
palette = [ctrl_color]


# Rate of change for each base compaction column, computed per replicate
# from the first and last timepoints, then summarized across replicates.
data = data.sort_values([subject, time_col])
g = data.groupby([subject])
first = g.first()
last = g.last()
dt = last[time_col] - first[time_col]

rates = (last[cmp_cols] - first[cmp_cols]).div(dt, axis=0)

summary = pd.DataFrame({
    'mean_rate': rates.mean(),
    'sd_rate': rates.std(),
})

print(summary)

## Control vs. drug-treatment compaction plots

Biological-replicate means of each compaction column are plotted across
timepoints for two condition sets:

1. DMSO vs. latrunculin-A.
2. DMSO vs. latrunculin-A vs. jasplakinolide.

Repeated-measures statistics are run per comparison and concatenated into
the combined stats table.

In [ ]:
ctrlvslat_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx'].isin(CTRLVSLAT_ORDER)]
ctrlvslat_cell_cmp_df = df_filt[df_filt['tx'].isin(CTRLVSLAT_ORDER)]
if 'omit' in ctrlvslat_cell_cmp_df.columns:
    ctrlvslat_cell_cmp_df = ctrlvslat_cell_cmp_df[ctrlvslat_cell_cmp_df['omit'] != 'Y']

prefix = 'ctrlvslat_cmp'
data = ctrlvslat_biorep_cmp_df
time_col = TIME_COL
subject = 'experiment'
group_category = 'tx'
hue_order = CTRLVSLAT_ORDER
palette = ctrllat_palette

for ycol in cmp_cols:
    comparison_label = {
        'columns': ycol,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}',
    }

    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False,
    )

    # Augment with per-timepoint Shapiro-Wilk on paired differences
    # (formal normality check) and Wilcoxon signed-rank (non-
    # parametric sensitivity check). Done before the append so the
    # saved stats table carries these columns too.
    stats_df = rsa.add_normality_checks(
        stats_df=stats_df,
        df_long=df_long,
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )
    stats_df = fo.tag_stats(stats_df, dataset=prefix.rstrip('_'),
                            group_variable=group_category,
                            groups=comparison_label['labels'])
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # LMM on cell-level data (between-cell tx, repeated within cell across time).
    stats_lmm = rsl.run_lmm_stats(
        df_cells=ctrlvslat_cell_cmp_df,
        subject=subject,
        cell_id=CELL_ID_COL,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False,
    )
    stats_lmm = fo.tag_stats(stats_lmm, dataset=prefix.rstrip('_'),
                             group_variable=group_category,
                             groups=comparison_label['labels'])
    stats_lmm_combined = pd.concat([stats_lmm_combined, stats_lmm], axis=0)

    fig_stats = stats_lmm if USE_LMM_FOR_FIGURES else stats_df

    # Render once with RM-ANOVA-derived annotations and once with
    # LMM-derived annotations, each saved to a distinct file.
    # Suppress annotation at non-significant timepoints; the printed
    # summary below each plot still reports every contrast.
    for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
        _graphname = f"{comparison_label['name']}_{_tag}"
        _fig_stats_for_plot = fo.filter_significant_rows(
            _stats, comparison_label['name'], rename_to=_graphname,
        )
        ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=palette,
        tx_line=0,
        figsize=timelapse_figsize,
            graphname=_graphname,
            save_dir=run_dir,
            stats_df=_fig_stats_for_plot,
        )
        print(f'\n[{_tag.upper()}]')
        fo.print_stats_summary(_stats, comparison_label)
    fo.save_plot_data(
        df_long,
        plot_data_dir / f"{comparison_label['name']}.csv",
        columns=[subject, time_col, group_category, ycol],
    )
    rsa.save_qq_plots(
        df_long=df_long,
        comparison_label=comparison_label,
        savepath=plot_data_dir / 'qq' / f"qq_{comparison_label['name']}.svg",
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )

    utils.safe_save_csv(stats_combined, stats_df_path)
    utils.safe_save_csv(stats_lmm_combined, stats_lmm_df_path)

In [ ]:
ctrlvslat_biorep_cmp_df = biorep_cmp_df[biorep_cmp_df['tx'].isin(DRUGTX_ORDER)]
drugtx_cell_cmp_df = df_filt[df_filt['tx'].isin(DRUGTX_ORDER)]
if 'omit' in drugtx_cell_cmp_df.columns:
    drugtx_cell_cmp_df = drugtx_cell_cmp_df[drugtx_cell_cmp_df['omit'] != 'Y']

prefix = 'ctrlvslat_cmp'
data = ctrlvslat_biorep_cmp_df
time_col = TIME_COL
subject = 'experiment'
group_category = 'tx'
hue_order = DRUGTX_ORDER
palette = drugtx_palette

for ycol in cmp_cols:
    comparison_label = {
        'columns': ycol,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}',
    }

    df_long, stats_df = rs.run_repeated_measures_stats(
        df=data,
        subject=subject,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False,
    )

    # Augment with per-timepoint Shapiro-Wilk on paired differences
    # (formal normality check) and Wilcoxon signed-rank (non-
    # parametric sensitivity check). Done before the append so the
    # saved stats table carries these columns too.
    stats_df = rsa.add_normality_checks(
        stats_df=stats_df,
        df_long=df_long,
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )
    stats_df = fo.tag_stats(stats_df, dataset=prefix.rstrip('_'),
                            group_variable=group_category,
                            groups=comparison_label['labels'])
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # LMM on cell-level data (3-level between-cell tx).
    stats_lmm = rsl.run_lmm_stats(
        df_cells=drugtx_cell_cmp_df,
        subject=subject,
        cell_id=CELL_ID_COL,
        time=time_col,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False,
    )
    stats_lmm = fo.tag_stats(stats_lmm, dataset=prefix.rstrip('_'),
                             group_variable=group_category,
                             groups=comparison_label['labels'])
    stats_lmm_combined = pd.concat([stats_lmm_combined, stats_lmm], axis=0)

    fig_stats = stats_lmm if USE_LMM_FOR_FIGURES else stats_df

    # Render once with RM-ANOVA-derived annotations and once with
    # LMM-derived annotations, each saved to a distinct file.
    # Suppress annotation at non-significant timepoints; the printed
    # summary below each plot still reports every contrast.
    for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
        _graphname = f"{comparison_label['name']}_{_tag}"
        _fig_stats_for_plot = fo.filter_significant_rows(
            _stats, comparison_label['name'], rename_to=_graphname,
        )
        ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=palette,
        tx_line=0,
        figsize=timelapse_figsize,
            graphname=_graphname,
            save_dir=run_dir,
            stats_df=_fig_stats_for_plot,
        )
        print(f'\n[{_tag.upper()}]')
        fo.print_stats_summary(_stats, comparison_label)
    fo.save_plot_data(
        df_long,
        plot_data_dir / f"{comparison_label['name']}.csv",
        columns=[subject, time_col, group_category, ycol],
    )
    rsa.save_qq_plots(
        df_long=df_long,
        comparison_label=comparison_label,
        savepath=plot_data_dir / 'qq' / f"qq_{comparison_label['name']}.svg",
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )

    utils.safe_save_csv(stats_combined, stats_df_path)
    utils.safe_save_csv(stats_lmm_combined, stats_lmm_df_path)

## Compaction-zone metrics

Per-compaction-zone measurements are loaded, the spurious -0.5 h
timepoint is collapsed onto -1 h, cells with more than two missing
timepoints are dropped, and timepoints are restricted to the [-1, 10] h
window. Per cell and timepoint, the mean compaction-zone area and the
number of compaction zones are computed; per-timepoint changes (and
cumulative changes) of these summaries are then derived. Biological-
replicate means feed the DMSO vs. latrunculin-A timecourse plots
below.

In [ ]:
cmp_zone_df = pd.read_csv(CMP_ZONE_DF_PATH)
cmp_zone_df.head()

In [ ]:
time_col = TIME_COL

cmp_zone_df.loc[cmp_zone_df[time_col] == -0.5, time_col] = -1
cmp_zone_df_filt = rd.filter_incomplete_data(cmp_zone_df, time_col, max_num_incomplete=2)
cmp_zone_df_filt = rd.select_timepoints(cmp_zone_df_filt, time_col, -1, 10)
cmp_zone_df_filt.head()

In [ ]:
# Per-cell, per-timepoint mean compaction-zone area and zone count.
groupbycols = ['tx', 'experiment', time_col]
sum_cmp_groupby = groupbycols + ['UID']

cmp_area_col = 'area (microns^2)'
sum_cmp_zone_df = cmp_zone_df_filt.groupby(sum_cmp_groupby, as_index=False).agg({
    cmp_area_col: ['mean', 'count'],
})

# Flatten the MultiIndex left by the multi-aggregation.
sum_cmp_zone_df.columns = [
    '_'.join(col).strip() if col[1] else col[0]
    for col in sum_cmp_zone_df.columns.values
]

cmp_zone_ycols = ['mean cmp zone area (μm²)', 'num cmp zones']
sum_cmp_zone_df = sum_cmp_zone_df.rename(columns={
    f'{cmp_area_col}_mean': cmp_zone_ycols[0],
    f'{cmp_area_col}_count': cmp_zone_ycols[1],
})

# Add per-timepoint change and cumulative-change columns.
sum_cmp_zone_df, cmp_zone_ycols = rd.compute_change_cols(
    sum_cmp_zone_df, cmp_zone_ycols, time_col=time_col,
)

sum_cmp_zone_df.head()

In [ ]:
biorep_sum_cmp_zone_df = rd.compute_means_by_biorep(sum_cmp_zone_df, groupbycols, cmp_zone_ycols)
biorep_sum_cmp_zone_df_path = DF_PATH.parent / 'biorep_sum_cmp_zone_df.csv'
utils.safe_save_csv(biorep_sum_cmp_zone_df, biorep_sum_cmp_zone_df_path)
biorep_sum_cmp_zone_df.head()

In [ ]:
prefix = 'ctrlvslatA'
data = biorep_sum_cmp_zone_df[biorep_sum_cmp_zone_df['tx'].isin(CTRLVSLAT_ORDER)]
ctrlvslat_cell_cmpzone_df = sum_cmp_zone_df[sum_cmp_zone_df['tx'].isin(CTRLVSLAT_ORDER)]

time_col = TIME_COL
cum_change_cmp_cols = [col for col in cmp_zone_ycols if 'cumulative change' in col]
hue_order = CTRLVSLAT_ORDER
subject = 'experiment'
time = time_col
group_category = 'tx'

for ycol in cum_change_cmp_cols:
    comparison_label = {
        'columns': ycol,
        'labels': hue_order,
        'name': f'{prefix}_{ptd.clean_column_name(ycol)}',
    }

    df_long, stats_df = rs.run_repeated_measures_stats(
        data, subject, time, group_category, ycol, comparison_label, melt_df=False,
    )
    # Augment with per-timepoint Shapiro-Wilk on paired differences
    # (formal normality check) and Wilcoxon signed-rank (non-
    # parametric sensitivity check). Done before the append so the
    # saved stats table carries these columns too.
    stats_df = rsa.add_normality_checks(
        stats_df=stats_df,
        df_long=df_long,
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )
    stats_df = fo.tag_stats(stats_df, dataset=prefix.rstrip('_'),
                            group_variable=group_category,
                            groups=comparison_label['labels'])
    stats_combined = pd.concat([stats_combined, stats_df], axis=0)

    # LMM on cell-level cmp-zone summary.
    stats_lmm = rsl.run_lmm_stats(
        df_cells=ctrlvslat_cell_cmpzone_df,
        subject=subject,
        cell_id=CELL_ID_COL,
        time=time,
        group_category=group_category,
        ycol=ycol,
        comparison_label=comparison_label,
        melt_df=False,
    )
    stats_lmm = fo.tag_stats(stats_lmm, dataset=prefix.rstrip('_'),
                             group_variable=group_category,
                             groups=comparison_label['labels'])
    stats_lmm_combined = pd.concat([stats_lmm_combined, stats_lmm], axis=0)

    fig_stats = stats_lmm if USE_LMM_FOR_FIGURES else stats_df

    # Render once with RM-ANOVA-derived annotations and once with
    # LMM-derived annotations, each saved to a distinct file.
    # Suppress annotation at non-significant timepoints; the printed
    # summary below each plot still reports every contrast.
    for _tag, _stats in [('rmanova', stats_df), ('lmm', stats_lmm)]:
        _graphname = f"{comparison_label['name']}_{_tag}"
        _fig_stats_for_plot = fo.filter_significant_rows(
            _stats, comparison_label['name'], rename_to=_graphname,
        )
        ptd.plot_timelapse_lines(
        df_long=df_long,
        xcol=time_col,
        ycol=ycol,
        hue=group_category,
        hue_order=hue_order,
        palette=ctrllat_palette,
        group_labels=None,
            graphname=_graphname,
            save_dir=run_dir,
            stats_df=_fig_stats_for_plot,
        )
        print(f'\n[{_tag.upper()}]')
        fo.print_stats_summary(_stats, comparison_label)
    fo.save_plot_data(
        df_long,
        plot_data_dir / f"{comparison_label['name']}.csv",
        columns=[subject, time_col, group_category, ycol],
    )
    rsa.save_qq_plots(
        df_long=df_long,
        comparison_label=comparison_label,
        savepath=plot_data_dir / 'qq' / f"qq_{comparison_label['name']}.svg",
        subject=subject,
        time=time_col,
        group=group_category,
        value=ycol,
    )

In [ ]:
utils.safe_save_csv(stats_combined, stats_df_path)
utils.safe_save_csv(stats_lmm_combined, stats_lmm_df_path)

## Provenance manifest

Writes a JSON manifest next to the figures that records, for this run:
the input CSV paths and SHA-256 hashes; every configuration parameter
declared in the config cell; the list of figure and table files
generated; the git commit of this analysis repository; and a UTC
timestamp. The manifest is intended to make every published figure
traceable back to a specific input file, code revision, and parameter
set.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def git_commit(repo_dir):
    try:
        return subprocess.check_output(
            ['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return None


# Output files generated by this notebook (figures + tables).
output_files = sorted(
    [str(p) for p in run_dir.glob('*.svg')]
    + [
        str(biorep_cmp_df_path),
        str(biorep_actin_df_path),
        str(biorep_actin_before_cmp_df_path),
        str(biorep_sum_cmp_zone_df_path),
        str(stats_df_path),
        str(stats_lmm_df_path),
    ]
)

manifest = {
    'notebook': NOTEBOOK_NAME,
    'inputs': {
        'df_path': {
            'path': str(DF_PATH),
            'sha256': file_sha256(DF_PATH),
        },
        'cmp_zone_df_path': {
            'path': str(CMP_ZONE_DF_PATH),
            'sha256': file_sha256(CMP_ZONE_DF_PATH),
        },
    },
    'run_id': RUN_ID,
    'run_dir': str(run_dir),
    'config': {
        'GRAPHS_DIRPATH': str(GRAPHS_DIRPATH),
        'TIME_COL': TIME_COL,
        'BASE_CMP_COLS': BASE_CMP_COLS,
        'CTRLVSLAT_ORDER': CTRLVSLAT_ORDER,
        'DRUGTX_ORDER': DRUGTX_ORDER,
        'CELL_ID_COL': CELL_ID_COL,
        'USE_LMM_FOR_FIGURES': USE_LMM_FOR_FIGURES,
    },
    'outputs': output_files,
    'git_commit': git_commit(Path.cwd()),
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}

manifest_path = run_dir / 'manifest.json'
with open(manifest_path, 'w') as fh:
    json.dump(manifest, fh, indent=2)

print(f'Wrote {manifest_path}')

## Export this notebook to Markdown

Renders the executed notebook (code, prose, and inline figure references)
to a Markdown file in `GRAPHS_DIRPATH`, alongside the SVG figures and the
manifest, so the figures and the analysis steps that produced them are
archived together.

In [ ]:
notebook_path = Path(NOTEBOOK_NAME).resolve()
subprocess.run(
    [
        'jupyter', 'nbconvert',
        '--to', 'markdown',
        str(notebook_path),
        '--output-dir', str(run_dir),
    ],
    check=True,
)